# Spark K8s Sandbox — Getting Started

This notebook walks through the core capabilities of the sandbox:

1. **SparkSession** — verify Iceberg and Delta extensions are loaded from `spark-defaults.conf`
2. **Delta Lake** — write, read, update, and time-travel a Delta table
3. **Iceberg** — write, read, and time-travel an Iceberg table via the `local` catalog
4. **Landing Zone** — template for reading files you uploaded from the dashboard

All data is written to `/data/warehouse/` on the shared PVC — the same volume your batch jobs read and write.

## 1 — SparkSession

`spark-defaults.conf` is baked into the image and registers both the Delta and Iceberg extensions automatically. You do **not** need to pass any extra `.config()` calls — `getOrCreate()` is enough.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GettingStarted")
    .getOrCreate()
)

spark.version

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Verify that both extensions are registered
exts = spark.conf.get("spark.sql.extensions")
print("Extensions:", exts)

# Verify the Iceberg catalog is configured
warehouse = spark.conf.get("spark.sql.catalog.local.warehouse")
print("Iceberg warehouse:", warehouse)

## 2 — Delta Lake

Delta is **path-based** — you point Spark at a directory and use `format("delta")`. No catalog registration needed.

Key features demonstrated:
- Write (overwrite)
- Read
- Upsert / merge (`MERGE INTO`)
- Time travel (`VERSION AS OF`)

In [ ]:
from delta.tables import DeltaTable

DELTA_PATH = "/data/warehouse/delta/employees"

# --- Initial write ---
data = [
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Marketing",   72000),
    (3, "Carol",   "Engineering", 88000),
    (4, "Dave",    "HR",          65000),
]
schema = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").save(DELTA_PATH)

print("Version 0 written")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# --- Upsert: give Engineering a 10 % raise ---
dt = DeltaTable.forPath(spark, DELTA_PATH)

updates = spark.createDataFrame(
    [(1, "Alice", "Engineering", 104500),
     (3, "Carol", "Engineering",  96800)],
    schema
)

dt.alias("t").merge(
    updates.alias("u"),
    "t.id = u.id"
).whenMatchedUpdateAll().execute()

print("Version 1 — after raise:")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# --- Time travel: read back version 0 ---
print("Version 0 (original salaries):")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH).show()

# --- Delta history ---
dt.history().select("version", "timestamp", "operation").show()

## 3 — Iceberg

Iceberg is **catalog-based**. The `local` catalog is pre-configured as a `HadoopCatalog` pointing to `/data/warehouse/iceberg`. Tables are addressed as `local.<database>.<table>`.

Key features demonstrated:
- Create table with DDL
- Insert and read
- Schema evolution (add a column)
- Time travel (`VERSION AS OF` / snapshot IDs)

In [ ]:
# --- Create namespace and table ---
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.sandbox")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.sandbox.products (
        product_id   INT,
        product_name STRING,
        category     STRING,
        price        DOUBLE
    )
    USING iceberg
""")

# --- Insert data (snapshot 1) ---
spark.sql("""
    INSERT INTO local.sandbox.products VALUES
        (1, 'Laptop',     'Electronics', 999.99),
        (2, 'Desk Chair', 'Furniture',   249.00),
        (3, 'Notebook',   'Stationery',    4.99)
""")

print("Snapshot 1:")
spark.table("local.sandbox.products").show()

In [ ]:
# --- Insert more rows (snapshot 2) ---
spark.sql("""
    INSERT INTO local.sandbox.products VALUES
        (4, 'Monitor',    'Electronics', 399.00),
        (5, 'Pen Set',    'Stationery',    8.49)
""")

# --- View snapshot history ---
spark.sql("SELECT snapshot_id, committed_at, operation FROM local.sandbox.products.snapshots").show(truncate=False)

In [ ]:
# --- Time travel to snapshot 1 (only first 3 products) ---
snapshots = spark.sql(
    "SELECT snapshot_id FROM local.sandbox.products.snapshots ORDER BY committed_at"
).collect()

first_snapshot = snapshots[0]["snapshot_id"]
print(f"Reading snapshot {first_snapshot}:")

spark.read \
    .option("snapshot-id", first_snapshot) \
    .table("local.sandbox.products") \
    .show()

In [ ]:
# --- Schema evolution: add a 'in_stock' column ---
spark.sql("ALTER TABLE local.sandbox.products ADD COLUMN in_stock BOOLEAN")

spark.sql("UPDATE local.sandbox.products SET in_stock = true WHERE product_id IN (1, 3, 4)")
spark.sql("UPDATE local.sandbox.products SET in_stock = false WHERE product_id IN (2, 5)")

spark.table("local.sandbox.products").show()

## 4 — Reading from the Landing Zone

Files uploaded via the dashboard appear at `/data/warehouse/landing/<filename>`. Update the path and format below to match whatever you uploaded.

In [ ]:
import os

LANDING = "/data/warehouse/landing"
files = os.listdir(LANDING)
print("Files in landing zone:", files)

In [ ]:
# Replace 'your_file.csv' with an actual filename from the cell above
FILENAME = "your_file.csv"

df_landing = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LANDING}/{FILENAME}")
)

df_landing.printSchema()
df_landing.show(5)

In [ ]:
# Write the landing file as a Delta table for faster subsequent reads
TABLE_NAME = FILENAME.replace(".csv", "").replace("-", "_").replace(" ", "_")
OUT_PATH = f"/data/warehouse/delta/{TABLE_NAME}"

df_landing.write.format("delta").mode("overwrite").save(OUT_PATH)

print(f"Written to Delta at {OUT_PATH}")
spark.read.format("delta").load(OUT_PATH).show(5)

## Cleanup (optional)

Run this cell to drop the Iceberg table created during this walkthrough.

In [ ]:
# spark.sql("DROP TABLE IF EXISTS local.sandbox.products")
# spark.sql("DROP NAMESPACE IF EXISTS local.sandbox")
print("Uncomment the lines above to clean up.")